# ch05 Bonus 11：Qwen3 实现

> 对照官方 `ch05/11_qwen3`
> **参考真实模型**：阿里 Qwen3（2024-2025）

## 一句话

Qwen3 在 Llama 基础上加了 **QK-Norm**（query/key 归一化）提升训练稳定性，并默认用 **GQA**。

## 相对 Llama 的独特改造

**QK-Norm**：在算注意力分数前，先对 Q 和 K 各做一次 RMSNorm，再乘一个可学习的 scale。这能防止某些头的 Q/K 范数过大导致 attention 熵过低（注意力过于尖锐），是 2024 年多个新模型（Qwen3、Gemma2、OLMo2 等）的共性改进。

> 其余结构（RoPE、RMSNorm、SwiGLU、GQA）与 Llama 基本一致。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Qwen3Attention(nn.Module):
    """Qwen3 注意力：GQA + QK-Norm（q/k 各做 RMSNorm 再乘可学习 scale）。"""

    def __init__(self, d_in, d_out, n_heads, n_kv_heads, head_dim):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim
        self.group_size = n_heads // n_kv_heads
        self.q_proj = nn.Linear(d_in, n_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(d_in, n_kv_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(d_in, n_kv_heads * head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * head_dim, d_out, bias=False)
        # QK-Norm：可学习的 scale（Qwen3 的关键改造）
        self.q_scale = nn.Parameter(torch.ones(head_dim))
        self.k_scale = nn.Parameter(torch.ones(head_dim))

    def forward(self, x):
        b, n, _ = x.shape
        q = self.q_proj(x).view(b, n, self.n_heads, self.head_dim)
        k = self.k_proj(x).view(b, n, self.n_kv_heads, self.head_dim)
        v = self.v_proj(x).view(b, n, self.n_kv_heads, self.head_dim)
        # ★ QK-Norm：RMSNorm 后乘 scale
        q = q * torch.rsqrt(q.pow(2).mean(-1, keepdim=True) + 1e-6) * self.q_scale
        k = k * torch.rsqrt(k.pow(2).mean(-1, keepdim=True) + 1e-6) * self.k_scale
        # GQA：扩展 kv 头
        k = k.repeat_interleave(self.group_size, dim=2)
        v = v.repeat_interleave(self.group_size, dim=2)
        q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
        attn = torch.softmax(q @ k.transpose(2,3) / self.head_dim**0.5, dim=-1)
        out = (attn @ v).transpose(1,2).reshape(b, n, self.n_heads * self.head_dim)
        return self.out_proj(out)


layer = Qwen3Attention(768, 768, n_heads=8, n_kv_heads=2, head_dim=96)
out = layer(torch.randn(2, 16, 768))
print(f"Qwen3 注意力输出: {tuple(out.shape)}")
print("\n💡 QK-Norm 让 q/k 范数受控，防止注意力熵过低（过度聚焦），训练更稳。")